## 과제: 로컬 모델을 활용한 RAG 시스템 구축

**개요** 

지난 과제는 기본적인 RAG 프로세스를 구축하였습니다. 가장 기본적인 모듈과 쉽게 구현할 수 있는 방식으로요!

하지만, 우리는 때로는 외부 모델을 사용하거나, Local LLM을 사용하기도 합니다. 

혹은, 더 나은 방식의 embedding 모델을 사용하기도 합니다. 

이번 과제는 지난 번 주어진 BASIC RAG 파이프라인에서 아래의 요구사항에 따라 주요 모듈을 변경하는 것이 목표입니다!

## Task 1: OpenAI LLM 과 Embedding 모델을 탈피하기!

여러분 OpenAI 의 임베딩 모델과 LLM 은 사용할 때마다 과금이 되고 있습니다.

이번에 여러분께 주어진 임무는 과금 0원의 RAG 를 만들어 보는 것입니다.

그러기 위해서는 아래의 요구사항을 충족하는 코드를 작성해야 합니다.

1. 임베딩 모델을 변경하기(임베딩 강의를 참고하여 본인의 상황에 맞는 무료 모델을 선택하세요)
2. LLM 모델을 변경하기(모델 강의를 참고하여 본인의 상황에 맞는 무료 모델을 선택하세요)

**실습데이터**

소프트웨어정책연구소(SPRi) - 2023년 12월호

- 저자: 유재흥(AI정책연구실 책임연구원), 이지수(AI정책연구실 위촉연구원)
- 링크: https://spri.kr/posts/view/23669
- 파일명: `SPRI_AI_Brief_2023년12월호_F.pdf`

**안내**

> 아래는 기본 코드 입니다. 아래 코드에서 여러분들이 변경하시면 됩니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 단계 1: 문서 로드(Load Documents)
loader = PDFPlumberLoader("SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Answer in Korean.

#Question: 
{question} 

#Context: 
{context} 

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Task 2: Text Splitter 선택

지금까지 `RecursiveCharacterTextSplitter` 만을 사용하여 텍스트를 분할하였습니다.

이제는 여러분이 선택한 Text Splitter 를 사용하여 문서를 분할해 보세요.

**참고**

`SemanticChunker` 를 꼭 적용해 보세요!

In [1]:
# Your Code Here

## Task 3. 스트림릿에 100% 무료로 동작하는 RAG 

여러분이 만든 100% 무료 RAG 를 스트림릿에 배포해 보세요!

코드는 별고의 `.py` 파일에 작성해 주세요!

**주요 기능**

- PDF 파일 업로드
- FAISS 벡터스토어에 데이터 저장
- 업로드한 파일 기반 Q&A 챗봇
- 유료 모델 사용금지!!